In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SelfAttention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.Wq = nn.Linear(hidden, hidden, bias=False)
        self.Wk = nn.Linear(hidden, hidden, bias=False)
        self.Wv = nn.Linear(hidden, hidden, bias=False)
        self.Wo = nn.Linear(hidden, hidden, bias=False)

    def forward(self, x, mask):
        # x: [B, T, H]
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(x.size(-1))
        scores = scores.masked_fill(mask == 0, -1e9)

        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        return self.Wo(out)


class FeedForward(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.fc1 = nn.Linear(hidden, hidden * 2)
        self.fc2 = nn.Linear(hidden * 2, hidden)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn = SelfAttention(hidden)
        self.ffn = FeedForward(hidden)

    def forward(self, x, mask):
        x = self.attn(x, mask)
        x = self.ffn(x)
        return x


class CStyleTransformerNER(nn.Module):
    def __init__(self, vocab_size, hidden, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden)
        self.encoder = TransformerBlock(hidden)
        self.classifier = nn.Linear(hidden, num_classes)

    def forward(self, input_ids, mask):
        x = self.embedding(input_ids)
        x = self.encoder(x, mask)
        logits = self.classifier(x)
        return logits


In [ ]:
!pip uninstall -y datasets pyarrow fsspec
!pip install datasets fsspec


Found existing installation: datasets 4.5.0
Uninstalling datasets-4.5.0:
  Successfully uninstalled datasets-4.5.0
Found existing installation: pyarrow 23.0.0
Uninstalling pyarrow-23.0.0:
  Successfully uninstalled pyarrow-23.0.0
Found existing installation: fsspec 2025.10.0
Uninstalling fsspec-2025.10.0:
  Successfully uninstalled fsspec-2025.10.0
  Using cached datasets-4.5.0-py3-none-any.whl.metadata (19 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached pyarrow-23.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
Using cached datasets-4.5.0-py3-none-any.whl (515 kB)
Using cached fsspec-2025.10.0-py3-none-any.whl (200 kB)
Using cached pyarrow-23.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (47.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2

In [ ]:
import datasets, pyarrow, fsspec
print(datasets.__version__)
print(pyarrow.__version__)
print(fsspec.__version__)


4.5.0
23.0.0
2025.10.0


In [ ]:
import huggingface_hub
huggingface_hub.login()

In [ ]:
!pip install --upgrade datasets


In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

def write_split(split, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for example in split:
            tokens = example["tokens"]
            ner_tags = example["ner_tags"]
            for tok, tag in zip(tokens, ner_tags):
                f.write(f"{tok} {tag}\n")
            f.write("\n")  # sentence boundary

write_split(dataset["train"], "train.txt")
write_split(dataset["validation"], "valid.txt")
write_split(dataset["test"], "test.txt")

RuntimeError: Dataset scripts are no longer supported, but found conll2003.py

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")
print(dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using the latest cached version of the dataset since conll2003 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'conll2003' at /root/.cache/huggingface/datasets/conll2003/conll2003/1.0.0/9a4d16a94f8674ba3466315300359b0acd891b68b6c8743ddf60b9c702adce98 (last modified on Sun Feb  8 04:44:17 2026).


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [ ]:
train_ds = dataset["train"]
val_ds   = dataset["validation"]
test_ds  = dataset["test"]


In [ ]:
label_names = dataset["train"].features["ner_tags"].feature.names
NUM_CLASSES = len(label_names)

print(label_names)
print(NUM_CLASSES)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
9


In [ ]:
print(dataset)
print(dataset["train"][0])


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


In [ ]:
!git clone https://github.com/DaveGamble/cJSON.git


Cloning into 'cJSON'...
remote: Enumerating objects: 4707, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 4707 (delta 84), reused 31 (delta 31), pack-reused 4581 (from 3)
Receiving objects: 100% (4707/4707), 2.57 MiB | 7.51 MiB/s, done.
Resolving deltas: 100% (3113/3113), done.


In [ ]:
train_ds[0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [ ]:
VOCAB_SIZE = 256      # must match C
MAX_LEN = 128
PAD_ID = 0
IGNORE_INDEX = -1
EPOCHS = 100
LR = 5e-4
HIDDEN_SIZE = 256

In [ ]:
def simple_hash(word, vocab_size=VOCAB_SIZE):
    h = 5381
    for c in word:
        h = ((h << 5) + h) + ord(c)  # h * 33 + c
    return h % vocab_size

def encode_word(text, max_len=MAX_LEN):
    tokens = text.split(" ")
    ids = [simple_hash(tok) for tok in tokens[:max_len]]
    return ids, len(ids)

def tokenize_and_align_labels(example):
    input_ids = []
    labels = []

    for word, tag in zip(example["tokens"], example["ner_tags"]):
        input_ids.append(simple_hash(word))
        labels.append(tag)

    # Truncate
    input_ids = input_ids[:MAX_LEN]
    labels = labels[:MAX_LEN]

    seq_len = len(input_ids)

    # Padding
    pad_len = MAX_LEN - seq_len
    input_ids += [PAD_ID] * pad_len
    labels += [IGNORE_INDEX] * pad_len

    attention_mask = [1] * seq_len + [0] * pad_len

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask
    }

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    remove_columns=dataset["train"].column_names
)
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "labels", "attention_mask"]
)



Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [ ]:
from torch.utils.data import DataLoader

train_subset = tokenized_dataset["train"].select(range(500))
test_subset = tokenized_dataset["test"].select(range(50))

print(f"Size of training subset: {len(train_subset)}")
print(f"Size of testing subset: {len(test_subset)}")

train_loader = DataLoader(
    train_subset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_subset,
    batch_size=16
)

print(f"Number of batches in train_loader: {len(train_loader)}")
print(f"Number of batches in test_loader: {len(test_loader)}")

In [ ]:
model = CStyleTransformerNER(VOCAB_SIZE, HIDDEN_SIZE, NUM_CLASSES)
print(model)

criterion = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"]
        labels = batch["labels"]
        mask = batch["attention_mask"]

        optimizer.zero_grad()

        logits = model(input_ids, mask)
        loss = criterion(
            logits.view(-1, NUM_CLASSES),
            labels.view(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} | Loss {total_loss/len(train_loader):.4f}")


CStyleTransformerNER(
  (embedding): Embedding(256, 256)
  (encoder): TransformerBlock(
    (attn): SelfAttention(
      (Wq): Linear(in_features=256, out_features=256, bias=False)
      (Wk): Linear(in_features=256, out_features=256, bias=False)
      (Wv): Linear(in_features=256, out_features=256, bias=False)
      (Wo): Linear(in_features=256, out_features=256, bias=False)
    )
    (ffn): FeedForward(
      (fc1): Linear(in_features=256, out_features=512, bias=True)
      (fc2): Linear(in_features=512, out_features=256, bias=True)
    )
  )
  (classifier): Linear(in_features=256, out_features=9, bias=True)
)
Epoch 0 | Loss 1.1918
Epoch 1 | Loss 0.7857
Epoch 2 | Loss 0.6980
Epoch 3 | Loss 0.6369
Epoch 4 | Loss 0.5748
Epoch 5 | Loss 0.5000
Epoch 6 | Loss 0.4581
Epoch 7 | Loss 0.4253
Epoch 8 | Loss 0.3640
Epoch 9 | Loss 0.3292
Epoch 10 | Loss 0.2948
Epoch 11 | Loss 0.2762
Epoch 12 | Loss 0.2470
Epoch 13 | Loss 0.2321
Epoch 14 | Loss 0.2121
Epoch 15 | Loss 0.1969
Epoch 16 | Loss 0.1987

In [ ]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=4f6451b47395b5f97d351f8e5a171e8d675f78e70a62a5026cb7e89d621c8018
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
import random
random.seed(42)

id2label = {
    0: "O",
    1: "B-PER",
    2: "I-PER",
    3: "B-ORG",
    4: "I-ORG",
    5: "B-LOC",
    6: "I-LOC",
    7: "B-MISC",
    8: "I-MISC"
}

model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"]
        labels = batch["labels"]
        mask = batch["attention_mask"]

        logits = model(input_ids, mask)
        preds = torch.argmax(logits, dim=-1)

        for i in range(input_ids.size(0)):  # batch loop
            true_seq = []
            pred_seq = []

            for j in range(input_ids.size(1)):  # token loop
                if labels[i, j].item() == -1:
                    continue   # ignore PAD (same as your C code)

                true_seq.append(id2label[labels[i, j].item()])
                pred_seq.append(id2label[preds[i, j].item()])

            y_true.append(true_seq)
            y_pred.append(pred_seq)


In [ ]:
from seqeval.metrics import precision_score, recall_score, f1_score,classification_report
import time

def argmax_logits(logits):
    # logits: [T, C]
    return torch.argmax(logits, dim=-1)

def measure_latency(model, input_ids, mask):
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t1 = time.time()

    with torch.no_grad():
        _ = model(input_ids, mask)

    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t2 = time.time()

    latency_ms = (t2 - t1) * 1000
    tokens_per_sec = input_ids.size(1) / (latency_ms / 1000)

    print(f"Latency: {latency_ms:.3f} ms")
    print(f"Throughput: {tokens_per_sec:.2f} tokens/sec")

def memory_footprint(model):
    param_bytes = sum(p.numel() * 4 for p in model.parameters())
    print(f"Model parameters: {param_bytes / (1024**2):.2f} MB")



print("=== CoNLL-2003 Entity-Level Evaluation ===")
print(classification_report(y_true, y_pred))

print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1-score :", f1_score(y_true, y_pred))

measure_latency(model, input_ids, mask)
memory_footprint(model)



=== CoNLL-2003 Entity-Level Evaluation ===
              precision    recall  f1-score   support

         LOC       0.16      0.12      0.14        43
        MISC       0.03      0.11      0.05        18
         ORG       0.00      0.00      0.00         2
         PER       0.06      0.05      0.05        83

   micro avg       0.06      0.08      0.07       146
   macro avg       0.06      0.07      0.06       146
weighted avg       0.08      0.08      0.08       146

Precision: 0.06321839080459771
Recall   : 0.07534246575342465
F1-score : 0.06875
Latency: 7.930 ms
Throughput: 16142.12 tokens/sec
Model parameters: 2.26 MB


In [ ]:
for i in range(5):
    print(f"--- Sentence {i+1} ---")
    print(f"Tokens: {test_ds[i]['tokens']}")
    print(f"Predicted Labels: {y_pred[i]}")
    print("\n")

--- Sentence 1 ---
Tokens: ['SOCCER', '-', 'JAPAN', 'GET', 'LUCKY', 'WIN', ',', 'CHINA', 'IN', 'SURPRISE', 'DEFEAT', '.']
Predicted Labels: ['I-PER', 'O', 'B-PER', 'O', 'O', 'B-MISC', 'O', 'O', 'B-LOC', 'O', 'O', 'O']


--- Sentence 2 ---
Tokens: ['Nadim', 'Ladki']
Predicted Labels: ['B-PER', 'I-LOC']


--- Sentence 3 ---
Tokens: ['AL-AIN', ',', 'United', 'Arab', 'Emirates', '1996-12-06']
Predicted Labels: ['B-ORG', 'B-PER', 'I-PER', 'B-PER', 'I-ORG', 'I-LOC']


--- Sentence 4 ---
Tokens: ['Japan', 'began', 'the', 'defence', 'of', 'their', 'Asian', 'Cup', 'title', 'with', 'a', 'lucky', '2-1', 'win', 'against', 'Syria', 'in', 'a', 'Group', 'C', 'championship', 'match', 'on', 'Friday', '.']
Predicted Labels: ['B-LOC', 'B-MISC', 'O', 'O', 'O', 'O', 'B-MISC', 'O', 'O', 'O', 'O', 'O', 'I-MISC', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'I-PER', 'O']


--- Sentence 5 ---
Tokens: ['But', 'China', 'saw', 'their', 'luck', 'desert', 'them', 'in', 'the', 'second', 'match', 'of', 'the'